In [1]:
pip install pandas scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

In [18]:
data = pd.read_csv("email.csv")

# Rename columns properly
data.columns = ['label', 'message']

print(data.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [19]:
# Normalize labels
data['label'] = data['label'].astype(str).str.lower().str.strip()

print("Before mapping:", data['label'].unique())

# Handle all possible formats
data['label'] = data['label'].replace({
    'ham': 0,
    'spam': 1,
    '0': 0,
    '1': 1
})

# Convert safely
data['label'] = pd.to_numeric(data['label'], errors='coerce')

# Remove invalid rows
data = data.dropna(subset=['label'])

# Convert to int
data['label'] = data['label'].astype(int)

print("\nAfter cleaning:")
print(data['label'].value_counts())
print("Total rows:", len(data))

Before mapping: ['ham' 'spam' '{"mode":"full"']

After cleaning:
label
0    4825
1     747
Name: count, dtype: int64
Total rows: 5572


In [20]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

data['message'] = data['message'].apply(clean_text)

In [21]:
data['length'] = data['message'].apply(len)

print("\nAverage Length:")
print(data.groupby('label')['length'].mean())


Average Length:
label
0     67.140104
1    116.384203
Name: length, dtype: float64


In [22]:
print("\nNull values:\n", data.isnull().sum())
print("Dataset size:", data.shape)


Null values:
 label      0
message    0
length     0
dtype: int64
Dataset size: (5572, 3)


In [23]:
X = data['message']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", len(X_train))
print("Testing size:", len(X_test))

Training size: 4457
Testing size: 1115


In [24]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=3000,
    ngram_range=(1, 2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [25]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

In [26]:
y_pred = model.predict(X_test_vec)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.9739910313901345

Confusion Matrix:
 [[966   0]
 [ 29 120]]


In [28]:
feature_names = vectorizer.get_feature_names_out()

# Use correct attribute
spam_probs = model.feature_log_prob_[1]   # 1 = spam class

# Get top 10 words
top_indices = spam_probs.argsort()[-10:]

print("\nTop Spam Words:")
for i in top_indices:
    print(feature_names[i])


Top Spam Words:
prize
ur
reply
new
claim
mobile
stop
text
txt
free


In [29]:
def predict_spam(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    
    prediction = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][prediction]
    
    return ("SPAM" if prediction == 1 else "NOT SPAM", prob)

In [30]:
msg = "You won a free gift card!"
label, confidence = predict_spam(msg)

print("Prediction:", label)
print("Confidence:", round(confidence, 2))

Prediction: SPAM
Confidence: 0.71
